In [3]:
!pip install onnx onnxruntime-gpu onnxsim


  Using cached onnxruntime_gpu-1.24.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.3 kB)
  Using cached onnxsim-0.4.36.tar.gz (21.0 MB)
  Preparing metadata (setup.py) ... done
Using cached onnxruntime_gpu-1.24.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (252.6 MB)
  Created wheel for onnxsim: filename=onnxsim-0.4.36-cp312-cp312-linux_x86_64.whl size=2200379 sha256=ae4b72b1e6e16f4a2563bc969d687c41776b643b7cb4baaf583dfaa42e080a39
  Stored in directory: /root/.cache/pip/wheels/73/5d/cc/db1350d9fabfe7f8442b5d97aff2ff543fc253277f71a6508f
Successfully built onnxsim


In [4]:
import torch
import torchvision.models as models
import numpy as np
import onnx
import onnxruntime as ort
import time
from tqdm import tqdm


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.eval()
model.to(device)

dummy = torch.randn(1, 3, 224, 224).to(device)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 213MB/s]


In [6]:
torch.onnx.export(
    model,
    dummy,
    "resnet50.onnx",
    opset_version=17,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch"},
        "output": {0: "batch"}
    }
)


/tmp/ipykernel_55/3451334201.py:1: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


In [7]:
onnx_model = onnx.load("resnet50.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model is valid")


ONNX model is valid


In [8]:
def benchmark_pytorch(model, device, runs=200):
    x = torch.randn(1,3,224,224).to(device)
    
    # warmup
    for _ in range(20):
        _ = model(x)
    
    torch.cuda.synchronize()
    start = time.time()
    
    for _ in range(runs):
        _ = model(x)
    
    torch.cuda.synchronize()
    end = time.time()
    
    return (end - start) / runs

pt_latency = benchmark_pytorch(model, device)
print("PyTorch Latency:", pt_latency*1000, "ms")


PyTorch Latency: 6.508196592330933 ms


In [9]:
session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

providers = ["CUDAExecutionProvider"] if device=="cuda" else ["CPUExecutionProvider"]

session = ort.InferenceSession(
    "resnet50.onnx",
    providers=providers,
    sess_options=session_options
)


In [10]:
def benchmark_onnx(session, runs=200):
    x = np.random.randn(1,3,224,224).astype(np.float32)
    
    # warmup
    for _ in range(20):
        session.run(None, {"input": x})
    
    start = time.time()
    
    for _ in range(runs):
        session.run(None, {"input": x})
    
    end = time.time()
    
    return (end - start) / runs

onnx_latency = benchmark_onnx(session)
print("ONNX Runtime Latency:", onnx_latency*1000, "ms")


ONNX Runtime Latency: 5.177910327911377 ms


In [14]:
def benchmark_batch(session, batch, runs=100):
    x = np.random.randn(batch,3,224,224).astype(np.float32)
    
    # warmup
    for _ in range(10):
        session.run(None, {"input": x})
    
    start = time.time()
    for _ in range(runs):
        session.run(None, {"input": x})
    end = time.time()
    
    return (end - start) / runs

for batch in [1,4,8,16]:
    latency = benchmark_batch(session, batch)
    print(f"Batch {batch}: {latency*1000:.2f} ms")


Batch 1: 5.36 ms
Batch 4: 9.82 ms
Batch 8: 16.32 ms
Batch 16: 28.85 ms


In [12]:
print("===== COMPARISON =====")
print(f"PyTorch latency: {pt_latency*1000:.2f} ms")
print(f"ONNX Runtime latency: {onnx_latency*1000:.2f} ms")


===== COMPARISON =====
PyTorch latency: 6.51 ms
ONNX Runtime latency: 5.18 ms


In [13]:
torch_mem = torch.cuda.memory_allocated() / 1024**2
print("GPU memory allocated (MB):", torch_mem)


GPU memory allocated (MB): 106.42724609375
